
# 🔁 Daily Repo Loop for **Ego4d-LiteSTA** (Colab-first)

This block sets up SSH to GitHub (**Deploy Key stored in Drive**), pulls your repo, runs code, logs outputs, and pushes logs/artifacts back to GitHub.
- One-time: add your **Deploy Key** public key to your GitHub repo Deploy Keys with **write access**.
- Edit the variables (**GITHUB_USER**, **REPO_NAME**) below once; they persist per notebook.


In [2]:
# ⛳ Mount Drive and set variables
from google.colab import drive
drive.mount('/content/drive')

GITHUB_USER = "saeedzns"          # <<< EDIT ME
REPO_NAME   = "Ego4d-LiteSTA"             # <<< default as requested
REPO_SSH    = f"git@github.com:{GITHUB_USER}/{REPO_NAME}.git"
WORKDIR     = f"/content/drive/MyDrive/{REPO_NAME}"
SSH_DIR     = "/content/drive/MyDrive/.ssh"

print("Repo:", REPO_SSH)
print("Workdir:", WORKDIR)


# 🔐 SSH + git config + clone/pull
# 1) Git identity (optional but recommended)
!git config --global user.name "saeedzns"
!git config --global user.email "zns1992@gmail.com"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo: git@github.com:saeedzns/Ego4d-LiteSTA.git
Workdir: /content/drive/MyDrive/Ego4d-LiteSTA


In [3]:
# Git keepalive + sane pack settings for Colab
import os, subprocess
os.environ['GIT_SSH_COMMAND'] = 'ssh -o ServerAliveInterval=30 -o ServerAliveCountMax=10'
subprocess.run(['git','config','--global','core.compression','9'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['git','config','--global','pack.threads','2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('Configured GIT_SSH_COMMAND and git pack settings')


Configured GIT_SSH_COMMAND and git pack settings


In [4]:
# 2) SSH agent and key from Drive (key filename: colab_egolite)
%%bash -s "$SSH_DIR"
SSH_DIR="$1"
mkdir -p "$SSH_DIR"
chmod 700 "$SSH_DIR"
eval "$(ssh-agent -s)"
ssh-add "$SSH_DIR/colab_egolite" >/dev/null 2>&1 || true
mkdir -p ~/.ssh && chmod 700 ~/.ssh
cat <<EOF > ~/.ssh/config
Host github.com
  HostName github.com
  User git
  IdentityFile $SSH_DIR/colab_egolite
  IdentitiesOnly yes
StrictHostKeyChecking accept-new
EOF
chmod 600 ~/.ssh/config
ssh-keyscan -t rsa,ed25519 github.com >> ~/.ssh/known_hosts 2>/dev/null || true
chmod 644 ~/.ssh/known_hosts

Agent pid 103922


## Pull and Push — Daily Sync
Run after setting variables and SSH above.


In [ ]:
%cd "$WORKDIR"
!bash pull.sh


/content/drive/MyDrive/Ego4d-LiteSTA
[pull] Fetching and pulling latest for 'main'...
From github.com:saeedzns/Ego4d-LiteSTA
 * branch            main       -> FETCH_HEAD
Already up to date.
[pull] Done.


In [ ]:
%cd "$WORKDIR"
!bash push.sh


/content/drive/MyDrive/Ego4d-LiteSTA
[push] Staging all changes...


## Fast Push (Colab)
Uses /content SSD and rsyncs code-only for faster commits/pushes.
- Requires prior setup cells (Drive + SSH).
- Pushes only code/config (heavy artifacts excluded).


In [41]:
%env FAST_PUSH_INCLUDE_HEAVY=1


env: FAST_PUSH_INCLUDE_HEAVY=1


In [58]:
WORKDIR = f"/content/drive/MyDrive/{REPO_NAME}"

In [60]:
%cd "$WORKDIR"
%env FAST_PUSH_INCLUDE_HEAVY=1
!FAST_PUSH=1 bash push.sh "colab fast push" "$WORKDIR"

/content/drive/MyDrive/Ego4d-LiteSTA
env: FAST_PUSH_INCLUDE_HEAVY=1
[push] Using fast Colab path via scripts/push_fast_colab.sh
WORKDIR does not look like a git repo: f"/content/drive/MyDrive/Ego4d-LiteSTA"


# Ego4D Annotation Transformation
This notebook helps you transform Ego4D annotations into different variants, e.g. with scaled down bboxes.

## Prerequisites
1. Use the [Ego4D CLI](https://ego4d-data.org/docs/start-here/) to download the annotations dataset.
2. Install all the packages in this notebook using `requirements.txt`.

## **Useful Links:**

[Ego4D Docs - Start Here!](https://ego4d-data.org/docs/start-here/#Download-The-CLI)

[Data Overview](https://ego4d-data.org/docs/data-overview/)

[Official Ego4D Site](https://ego4d-data.org/)

In [3]:
# Set your options here

import os

CLI_OUTPUT_DIR = "/content/drive/MyDrive/ego4d_data" # Replace with the full path to the --output_directory you pass to the cli
VERSION = "v2"

METADATA_PATH = os.path.join(CLI_OUTPUT_DIR, "ego4d.json")
ANNOTATIONS_PATH = os.path.join(CLI_OUTPUT_DIR, VERSION, "annotations")

assert os.path.exists(METADATA_PATH), f"Metadata doesn't exist at {METADATA_PATH}. Is the CLI_OUTPUT_DIR right? Do you satisfy the pre-requisites?"
assert os.path.exists(os.path.join(ANNOTATIONS_PATH, "manifest.csv")), "Annotation metadata doesn't exist. Did you download it with the CLI?"

In [4]:
# Imports and consts
import random
import simplejson as json

from collections import namedtuple
from typing import List

annotation_files = [
    "av_train.json",
    "av_val.json",

    "fho_hands_train.json",
    "fho_hands_val.json",
    # "fho_lta_taxonomy.json",
    "fho_lta_train.json",
    "fho_lta_val.json",
    "fho_scod_train.json",
    "fho_scod_val.json",
    "fho_sta_train.json",
    "fho_sta_val.json",

    # "manifest.csv",
    # "manifest.ver",

    "moments_train.json",
    "moments_val.json",

    "narration.json",
    # "narration_noun_taxonomy.csv",
    # "narration_verb_taxonomy.csv",

    "nlq_train.json",
    "nlq_val.json",
    "vq_train.json",
    "vq_val.json"
]

def load_json_from_path(path):
    with open(path) as json_file:
        return json.load(json_file)

# Load Metadata Into Memory
First load video metadata into a dict. This is useful for mappers/selectors that need video resolution or other info.

In [8]:
# Load ego4D video metadata into a dictionary for easy indexing
meta = load_json_from_path(METADATA_PATH)

metadata = {
    video['video_uid']: {
        **{
            k: v
            for k, v in video.items()
                if k != 'video_metadata'
        },
        **video['video_metadata']
    }
    for video in meta['videos']
}

print(len(metadata))
print(f"Keys Accessible in Metadata: {list(metadata[list(metadata.keys())[0]].keys())}")

9821
Keys Accessible in Metadata: ['video_uid', 'duration_sec', 'scenarios', 'is_grouped', 'group_video_uids', 'split_em', 'split_av', 'split_fho', 'split_goalstep', 's3_path', 'manifold_path', 'origin_video_id', 'video_source', 'device', 'physical_setting_name', 'fb_participant_id', 'is_stereo', 'has_imu', 'has_gaze', 'imu_metadata', 'gaze_metadata', 'video_components', 'concurrent_sets', 'has_redacted_regions', 'redacted_intervals', 'gaps', 'fps', 'num_frames', 'video_codec', 'audio_codec', 'display_resolution_width', 'display_resolution_height', 'sample_resolution_width', 'sample_resolution_height', 'mp4_duration_sec', 'video_start_sec', 'video_duration_sec', 'audio_start_sec', 'audio_duration_sec', 'video_start_pts', 'video_duration_pts', 'video_base_numerator', 'video_base_denominator', 'audio_start_pts', 'audio_duration_pts', 'audio_base_numerator', 'audio_base_denominator']


# Define JSON Transformation Utils
We transform the annotation jsons with a selector/mapper architecture. The selector gets a json object and decides whether it needs to be changed. If it does, then the mapper takes in that object and returns a transformed one to take its place.

We call each selector/mapper pair a 'Transform'.

We input an ordered list of Transforms to be applied; only the first matching Transform for each object is used.

In [5]:
# Applies a list of Transforms to the given json object
# using a recursive DFS
#
# Each Transform is a selector and a mapper. The selector
# identifies json objects that should be transformed, and
# the mapper transforms them.
#
# Selectors have type: (obj: any, context: dict) -> bool
# Mappers have type: (obj: any, context: dict) -> any
#
# Context fields are passed into the selector/mapper for
# all children once they're seen. See wiki for schema
# structure and add any context fields you need to use
# below.

Transform = namedtuple('Transformation', 'selector mapper')
context_fields = ['video_uid', 'video_id', 'clip_uid']

def _apply_transforms(obj, transforms, context=None):
    context = context or {}

    for transform in transforms:
        selector, mapper = transform
        if selector(obj, context):
            return mapper(obj, context)

    if type(obj) is dict:
#       Context fields are propagated down to their children in
#       the context object, so mappers/selectors can use them
        context = {
            **context,
            **{
                k: obj[k]
                for k in context_fields
                    if k in obj
            }
        }

        return {
            k: _apply_transforms(v, transforms, {**context, 'key': k})
            for k, v in obj.items()
        }
    elif type(obj) is list:
        return [
            _apply_transforms(v, transforms, context)
            for v in obj
        ]
    return obj

def transform_annotations(input_path: str, output_path: str, transforms: List[Transform]):
    print(f"\nloading {input_path}...")
    original_obj = load_json_from_path(input_path)
    print("transforming...")
    transformed_obj = _apply_transforms(original_obj, transforms)
    print(f"writing {output_path}...")
    with open(output_path, 'w') as f:
        json.dump(transformed_obj, f)
    print("done.")


# Define Transforms
Here are a few sample Transforms.

In [6]:
def scale_ratio(video_uid, new_height):
    return float(new_height) / metadata[video_uid]['display_resolution_height']

# dummy example, increments all floats/ints by 1
increment_nums = Transform(
    lambda obj, ctx: isinstance(obj, int) or isinstance(obj, float), # Selector
    lambda obj, ctx: obj + 1 # Mapper
)

# scale bboxes - works for the schemas of av, fho_scod, and vq
def scale_bboxes(new_height): return Transform(
    lambda obj, ctx: type(obj) is dict and 'x' in obj and 'y' in obj and 'width' in obj and 'height' in obj, # Selector
    lambda obj, ctx: { # Mapper
        **obj, # av has extra properties to retain, e.g. person id
        'x': obj['x'] * scale_ratio(ctx['video_uid'], new_height),
        'y': obj['y'] * scale_ratio(ctx['video_uid'], new_height),
        'width': obj['width'] * scale_ratio(ctx['video_uid'], new_height),
        'height': obj['height'] * scale_ratio(ctx['video_uid'], new_height),
    }
)

# scale bboxes - works for the schema of fho_sta
def fho_sta_scale_boxes(new_height): return Transform(
    lambda obj, ctx: type(obj) is list and ctx.get('key') == 'box', # Selector
    lambda obj, ctx: [ # Mapper
        point * scale_ratio(ctx['video_uid'], new_height)
        for point in obj
    ]
)

# Execute transformations and output new annotation files
Now we apply these Transforms to the original annotation files and create new ones.

In [15]:
# transform_jobs has format: [
#    ( input file (in annotations directory), output filename, list of Transforms to apply ), ...
# ]
transform_jobs = [
    ('av_train.json', 'av_train_height-540.json', [scale_bboxes(540)]),
    ('av_val.json', 'av_val_height-540.json', [scale_bboxes(540)]),
    ('fho_hands_train.json', 'fho_hands_train_height-540.json', [scale_bboxes(540)]),
    ('fho_hands_val.json', 'fho_hands_val_height-540.json', [scale_bboxes(540)]),
    ('fho_scod_train.json', 'fho_scod_train_height-540.json', [scale_bboxes(540)]),
    ('fho_scod_val.json', 'fho_scod_val_height-540.json', [scale_bboxes(540)]),
    ('vq_train.json', 'vq_train_height-540.json', [scale_bboxes(540)]),
    ('vq_val.json', 'vq_val_height-540.json', [scale_bboxes(540)]),

    ('fho_sta_train.json', 'fho_sta_train_height-540.json', [fho_sta_scale_boxes(540)]),
    ('fho_sta_val.json', 'fho_sta_val_height-540.json', [fho_sta_scale_boxes(540)]),
]

output_dir = "/content/drive/MyDrive/ego4d_data/v2/annotations_540ss"
os.makedirs(output_dir, exist_ok=True)

for j in transform_jobs:
    input_path = os.path.join(ANNOTATIONS_PATH, j[0])
    output_path = os.path.join(output_dir, j[1])
    if os.path.exists(input_path):
        transform_annotations(input_path, output_path, j[2])
    else:
        print(f"Skipping missing file: {input_path}")


loading /content/drive/MyDrive/ego4d_data/v2/annotations/av_train.json...
transforming...
writing /content/drive/MyDrive/ego4d_data/v2/annotations_540ss/av_train_height-540.json...
done.

loading /content/drive/MyDrive/ego4d_data/v2/annotations/av_val.json...
transforming...
writing /content/drive/MyDrive/ego4d_data/v2/annotations_540ss/av_val_height-540.json...
done.
Skipping missing file: /content/drive/MyDrive/ego4d_data/v2/annotations/fho_hands_train.json
Skipping missing file: /content/drive/MyDrive/ego4d_data/v2/annotations/fho_hands_val.json
Skipping missing file: /content/drive/MyDrive/ego4d_data/v2/annotations/fho_scod_train.json
Skipping missing file: /content/drive/MyDrive/ego4d_data/v2/annotations/fho_scod_val.json

loading /content/drive/MyDrive/ego4d_data/v2/annotations/vq_train.json...
transforming...
writing /content/drive/MyDrive/ego4d_data/v2/annotations_540ss/vq_train_height-540.json...
done.

loading /content/drive/MyDrive/ego4d_data/v2/annotations/vq_val.json...
t

# Test Outputs

Always validate the annotation output before using it. We look at a specific json path to verify that the bboxes have been scaled appropriately, then do a quick 'deep diff' to catch high-level changes.

In [7]:
# Manual json path checks

output_dir = "/content/drive/MyDrive/ego4d_data/v2/annotations_540ss"
old_av, new_av = load_json_from_path(os.path.join(ANNOTATIONS_PATH, 'av_train.json')), load_json_from_path(os.path.join(output_dir, 'av_train_height-540.json'))
f = lambda x: x['videos'][0]['clips'][0]['persons'][1]['tracking_paths'][0]['track'][0]

print(old_av['videos'][0]['video_uid'])
print(f(old_av), "\n", f(new_av))
# len(old_av['videos'][0]['clips'][0]['persons'][0]['tracking_paths'])

733ac083-1962-46c3-9a46-1efee74fee90
{'x': 561.51, 'y': 818.64, 'width': 246.89, 'height': 246.89, 'frame': 0, 'video_frame': 2430, 'clip_frame': None} 
 {'x': 280.755, 'y': 409.32, 'width': 123.445, 'height': 123.445, 'frame': 0, 'video_frame': 2430, 'clip_frame': None}


In [10]:
!pip install deepdiff

In [12]:
# Deepdiffs
# 'No change' results are not reliable when using the max_diffs arg. 'Changed' results are.
# max_diffs arg is usually required to finish diffing in a reasonable amount of time

# transform_jobs has format: [
#    ( input file (in annotations directory), output filename, list of Transforms to apply ), ...
# ]
transform_jobs = [
    ('av_train.json', 'av_train_height-540.json', [scale_bboxes(540)]),
    ('av_val.json', 'av_val_height-540.json', [scale_bboxes(540)]),
    ('fho_hands_train.json', 'fho_hands_train_height-540.json', [scale_bboxes(540)]),
    ('fho_hands_val.json', 'fho_hands_val_height-540.json', [scale_bboxes(540)]),
    ('fho_scod_train.json', 'fho_scod_train_height-540.json', [scale_bboxes(540)]),
    ('fho_scod_val.json', 'fho_scod_val_height-540.json', [scale_bboxes(540)]),
    ('vq_train.json', 'vq_train_height-540.json', [scale_bboxes(540)]),
    ('vq_val.json', 'vq_val_height-540.json', [scale_bboxes(540)]),

    ('fho_sta_train.json', 'fho_sta_train_height-540.json', [fho_sta_scale_boxes(540)]),
    ('fho_sta_val.json', 'fho_sta_val_height-540.json', [fho_sta_scale_boxes(540)]),
]

from deepdiff import DeepDiff
from pprint import pprint, pformat

def print_json_file_diff(a, b, print_limit=5000):
    print(f"loading {a}, {b}...")
    obj_a, obj_b = load_json_from_path(a), load_json_from_path(b)
    print("diffing...")
    x = DeepDiff(obj_a, obj_b, max_diffs=500)
    print("pformatting...")
    y = pformat(x)
    print("printing...")
    print(y[:print_limit])

output_dir = "/content/drive/MyDrive/ego4d_data/v2/annotations_540ss"

for j in transform_jobs:
    input_path = os.path.join(ANNOTATIONS_PATH, j[0])
    output_path = os.path.join(output_dir, j[1])
    if os.path.exists(input_path) and os.path.exists(output_path):
        print_json_file_diff(input_path, output_path)
    else:
        print(f"Skipping diff for missing file(s): {input_path} or {output_path}")

loading /content/drive/MyDrive/ego4d_data/v2/annotations/av_train.json, /content/drive/MyDrive/ego4d_data/v2/annotations_540ss/av_train_height-540.json...
diffing...


pformatting...
printing...
{}
loading /content/drive/MyDrive/ego4d_data/v2/annotations/av_val.json, /content/drive/MyDrive/ego4d_data/v2/annotations_540ss/av_val_height-540.json...


diffing...
pformatting...
printing...
{}
Skipping diff for missing file(s): /content/drive/MyDrive/ego4d_data/v2/annotations/fho_hands_train.json or /content/drive/MyDrive/ego4d_data/v2/annotations_540ss/fho_hands_train_height-540.json
Skipping diff for missing file(s): /content/drive/MyDrive/ego4d_data/v2/annotations/fho_hands_val.json or /content/drive/MyDrive/ego4d_data/v2/annotations_540ss/fho_hands_val_height-540.json
Skipping diff for missing file(s): /content/drive/MyDrive/ego4d_data/v2/annotations/fho_scod_train.json or /content/drive/MyDrive/ego4d_data/v2/annotations_540ss/fho_scod_train_height-540.json
Skipping diff for missing file(s): /content/drive/MyDrive/ego4d_data/v2/annotations/fho_scod_val.json or /content/drive/MyDrive/ego4d_data/v2/annotations_540ss/fho_scod_val_height-540.json
loading /content/drive/MyDrive/ego4d_data/v2/annotations/vq_train.json, /content/drive/MyDrive/ego4d_data/v2/annotations_540ss/vq_train_height-540.json...
diffing...


pformatting...
printing...
{'values_changed': {"root['videos'][0]['clips'][0]['annotations'][0]['query_sets']['1']['response_track'][0]['height']": {'new_value': 20.7,
                                                                                                                          'old_value': 41.4},
                    "root['videos'][0]['clips'][0]['annotations'][0]['query_sets']['1']['response_track'][0]['width']": {'new_value': 5.6,
                                                                                                                         'old_value': 11.2},
                    "root['videos'][0]['clips'][0]['annotations'][0]['query_sets']['1']['response_track'][0]['x']": {'new_value': 464.84,
                                                                                                                     'old_value': 929.68},
                    "root['videos'][0]['clips'][0]['annotations'][0]['query_sets']['1']['response_track'][0]['y']": {'new_value': 270

diffing...
pformatting...
printing...
{'values_changed': {"root['videos'][0]['clips'][0]['annotations'][0]['query_sets']['3']['response_track'][0]['height']": {'new_value': 35.345,
                                                                                                                          'old_value': 70.69},
                    "root['videos'][0]['clips'][0]['annotations'][0]['query_sets']['3']['response_track'][0]['width']": {'new_value': 58.215,
                                                                                                                         'old_value': 116.43},
                    "root['videos'][0]['clips'][0]['annotations'][0]['query_sets']['3']['response_track'][0]['x']": {'new_value': 0.55,
                                                                                                                     'old_value': 1.1},
                    "root['videos'][0]['clips'][0]['annotations'][0]['query_sets']['3']['response_track'][0]['y']": {'n

pformatting...
printing...
{}
loading /content/drive/MyDrive/ego4d_data/v2/annotations/fho_sta_val.json, /content/drive/MyDrive/ego4d_data/v2/annotations_540ss/fho_sta_val_height-540.json...
diffing...


pformatting...
printing...
{}


## History Cleanup: remove awscliv2.zip (optional)
This rewrites repo history to drop the large awscliv2.zip blobs.
- Runs on /content for speed.
- Force-push affects all collaborators; ensure you are the only active pusher.
- After success, consider recloning locally to avoid dangling objects.


In [65]:
%%bash -s "$WORKDIR"
set -euo pipefail
WORKDIR="$1"
if [ -z "$WORKDIR" ]; then echo "Set WORKDIR earlier in the notebook." >&2; exit 1; fi
origin=$(git -C "$WORKDIR" config --get remote.origin.url)
branch=$(git -C "$WORKDIR" rev-parse --abbrev-ref HEAD)
mirror=/content/egolite_cleanup.git
python3 -m pip -q install git-filter-repo
rm -rf "$mirror"
git clone --mirror "$origin" "$mirror"
cd "$mirror"
git filter-repo --invert-paths --path awscliv2.zip --force
git count-objects -vH
echo Origin: $origin
echo Branch: $branch
echo 'Review above size. If okay, remove # below to force-push:'
# git push --force --prune

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 4.1 MB/s eta 0:00:00
NOTICE: Removing 'origin' remote; see 'Why is my origin removed?'
        in the manual if you want to push back there.
        (was git@github.com:saeedzns/Ego4d-LiteSTA.git)
Parsed 42 commits
New history written in 0.34 seconds; now repacking/cleaning...
Repacking your repo and cleaning out old unneeded objects
Completely finished after 0.63 seconds.
count: 0
size: 0 bytes
in-pack: 8906
packs: 1
size-pack: 4.56 MiB
prune-packable: 0
garbage: 0
size-garbage: 0 bytes
Origin: git@github.com:saeedzns/Ego4d-LiteSTA.git
Branch: main
Review above size. If okay, remove # below to force-push:


Cloning into bare repository '/content/egolite_cleanup.git'...
Note: A branch outside the refs/remotes/ hierarchy was not removed;
to delete it, use:
  git branch -d main


In [66]:
!git push --force --prune

Everything up-to-date
